In [39]:
!pip install -q datasets huggingface_hub scipy numpy pandas tqdm

import re
import random
import numpy as np
import pandas as pd
from scipy import sparse
from collections import Counter
from tqdm.auto import tqdm
from datasets import load_dataset

random.seed(42)
np.random.seed(42)

In [40]:
DATASET_NAME = "ai4bharat/IndicCorpV2"
CONFIG_NAME = "indiccorp_v2"   # fixed config name — languages are SPLITS, not configs

LANGUAGE_SPLITS = [
    "asm_Beng", "ben_Beng", "brx_Deva", "doi_Deva", "gom_Deva",
    "guj_Gujr", "hin_Deva", "kan_Knda", "kas_Arab", "mai_Deva",
    "mal_Mlym", "mar_Deva", "mni_Mtei", "npi_Deva", "ory_Orya",
    "pan_Guru", "san_Deva", "snd_Deva", "tam_Taml", "tel_Telu",
    "urd_Arab", "khasi", "santhali"
]

print(f"Total languages: {len(LANGUAGE_SPLITS)}")
print(LANGUAGE_SPLITS)

Total languages: 23
['asm_Beng', 'ben_Beng', 'brx_Deva', 'doi_Deva', 'gom_Deva', 'guj_Gujr', 'hin_Deva', 'kan_Knda', 'kas_Arab', 'mai_Deva', 'mal_Mlym', 'mar_Deva', 'mni_Mtei', 'npi_Deva', 'ory_Orya', 'pan_Guru', 'san_Deva', 'snd_Deva', 'tam_Taml', 'tel_Telu', 'urd_Arab', 'khasi', 'santhali']


In [41]:
SENTENCE_END = re.compile(r'(?<=[.!?।॥؟۔])\s+')

def split_sentences(text, min_chars=15, min_words=4):
    text = re.sub(r'\s+', ' ', text.strip())
    if not text:
        return []
    parts = SENTENCE_END.split(text)
    sents = []
    for s in parts:
        s = s.strip()
        if len(s) >= min_chars and len(s.split()) >= min_words:
            sents.append(s)
    return sents

In [42]:
N_PER_LANG = 700
MAX_DOCS_SCAN = 30000  # safety cap while streaming

def collect_sentences_for_lang(lang_split, n_needed=N_PER_LANG):
    ds = load_dataset(
        DATASET_NAME,
        CONFIG_NAME,
        split=lang_split,
        streaming=True
    )
    sentences, seen = [], set()
    for i, ex in enumerate(ds):
        if i >= MAX_DOCS_SCAN or len(sentences) >= n_needed:
            break
        text = ex.get("text", "")
        for sent in split_sentences(text):
            if len(sentences) >= n_needed:
                break
            if sent in seen:
                continue
            seen.add(sent)
            sentences.append(sent)
    return sentences

all_rows = []
for lang in tqdm(LANGUAGE_SPLITS, desc="Languages"):
    try:
        sents = collect_sentences_for_lang(lang)
        if len(sents) < N_PER_LANG:
            print(f"WARNING: {lang} only yielded {len(sents)} sentences")
        for s in sents[:N_PER_LANG]:
            all_rows.append({"text": s, "label": lang})
    except Exception as e:
        print(f"FAILED for {lang}: {e}")

df = pd.DataFrame(all_rows)
print(df.shape)
print(df['label'].value_counts())

Languages:   0%|          | 0/23 [00:00<?, ?it/s]

(16100, 2)
label
asm_Beng    700
ben_Beng    700
brx_Deva    700
doi_Deva    700
gom_Deva    700
guj_Gujr    700
hin_Deva    700
kan_Knda    700
kas_Arab    700
mai_Deva    700
mal_Mlym    700
mar_Deva    700
mni_Mtei    700
npi_Deva    700
ory_Orya    700
pan_Guru    700
san_Deva    700
snd_Deva    700
tam_Taml    700
tel_Telu    700
urd_Arab    700
khasi       700
santhali    700
Name: count, dtype: int64


In [43]:
df.to_csv('/content/lang_id_data.csv', index=False)
print("Saved!")

Saved!


In [44]:
def stratified_split(df, train_frac=0.8, val_frac=0.1, seed=42):
    rng = np.random.RandomState(seed)
    train_parts, val_parts, test_parts = [], [], []
    for label, group in df.groupby('label'):
        idx = group.index.to_numpy().copy()
        rng.shuffle(idx)
        n = len(idx)
        n_train = int(round(n * train_frac))
        n_val = int(round(n * val_frac))
        train_parts.append(df.loc[idx[:n_train]])
        val_parts.append(df.loc[idx[n_train:n_train+n_val]])
        test_parts.append(df.loc[idx[n_train+n_val:]])
    train_df = pd.concat(train_parts).sample(frac=1, random_state=seed).reset_index(drop=True)
    val_df   = pd.concat(val_parts).sample(frac=1, random_state=seed).reset_index(drop=True)
    test_df  = pd.concat(test_parts).sample(frac=1, random_state=seed).reset_index(drop=True)
    return train_df, val_df, test_df

train_df, val_df, test_df = stratified_split(df)
print(train_df.shape, val_df.shape, test_df.shape)
print(train_df['label'].value_counts().head())

(12880, 2) (1610, 2) (1610, 2)
label
hin_Deva    560
urd_Arab    560
asm_Beng    560
khasi       560
mar_Deva    560
Name: count, dtype: int64


In [55]:
labels = sorted(df['label'].unique())
label2idx = {l: i for i, l in enumerate(labels)}
idx2label = {i: l for l, i in label2idx.items()}
N_CLASSES = len(labels)

y_train = train_df['label'].map(label2idx).to_numpy()
y_val   = val_df['label'].map(label2idx).to_numpy()
y_test  = test_df['label'].map(label2idx).to_numpy()

In [54]:
class CustomTfidfVectorizer:
    def __init__(self, max_features=40000, min_df=2,
                 word_ngram_range=(1, 2), char_ngram_range=(2, 4)):
        self.max_features = max_features
        self.min_df = min_df
        self.word_ngram_range = word_ngram_range
        self.char_ngram_range = char_ngram_range
        self.vocabulary_ = {}
        self.idf_ = None

    def _word_ngrams(self, tokens):
        feats = []
        n_min, n_max = self.word_ngram_range
        for n in range(n_min, n_max + 1):
            for i in range(len(tokens) - n + 1):
                feats.append("W_" + "▁".join(tokens[i:i+n]))
        return feats

    def _char_ngrams(self, text):
        feats = []
        n_min, n_max = self.char_ngram_range
        padded = f" {text} "
        for n in range(n_min, n_max + 1):
            for i in range(len(padded) - n + 1):
                feats.append("C_" + padded[i:i+n])
        return feats

    def _extract(self, text):
        tokens = text.split()
        return self._word_ngrams(tokens) + self._char_ngrams(text)

    def fit(self, texts):
        doc_freq = Counter()
        for text in tqdm(texts, desc="Building vocab (doc freq)"):
            for f in set(self._extract(text)):
                doc_freq[f] += 1

        filtered = {f: c for f, c in doc_freq.items() if c >= self.min_df}
        top_feats = sorted(filtered.items(), key=lambda x: -x[1])[:self.max_features]
        self.vocabulary_ = {f: i for i, (f, _) in enumerate(top_feats)}

        n_docs = len(texts)
        self.idf_ = np.zeros(len(self.vocabulary_), dtype=np.float64)
        for f, i in self.vocabulary_.items():
            self.idf_[i] = np.log(n_docs / (1 + filtered[f])) + 1.0
        return self

    def transform(self, texts):
        rows, cols, data = [], [], []
        for r, text in enumerate(tqdm(texts, desc="Transforming")):
            counts = Counter(self._extract(text))
            total = sum(counts.values()) or 1
            for f, c in counts.items():
                col = self.vocabulary_.get(f)
                if col is not None:
                    tf = c / total
                    rows.append(r)
                    cols.append(col)
                    data.append(tf * self.idf_[col])
        mat = sparse.csr_matrix((data, (rows, cols)),
                                 shape=(len(texts), len(self.vocabulary_)), dtype=np.float64)
        norms = np.sqrt(np.asarray(mat.multiply(mat).sum(axis=1))).ravel()
        norms[norms == 0] = 1.0
        inv_norms = sparse.diags(1.0 / norms)
        return (inv_norms @ mat).tocsr()

    def fit_transform(self, texts):
        self.fit(texts)
        return self.transform(texts)

In [47]:
vectorizer = CustomTfidfVectorizer(max_features=40000, min_df=2,
                                    word_ngram_range=(1, 2), char_ngram_range=(2, 4))

X_train = vectorizer.fit_transform(train_df['text'].tolist())
X_val   = vectorizer.transform(val_df['text'].tolist())
X_test  = vectorizer.transform(test_df['text'].tolist())

print("Vocab size:", len(vectorizer.vocabulary_))
print(X_train.shape, X_val.shape, X_test.shape)

Building vocab (doc freq):   0%|          | 0/12880 [00:00<?, ?it/s]

Transforming:   0%|          | 0/12880 [00:00<?, ?it/s]

Transforming:   0%|          | 0/1610 [00:00<?, ?it/s]

Transforming:   0%|          | 0/1610 [00:00<?, ?it/s]

Vocab size: 40000
(12880, 40000) (1610, 40000) (1610, 40000)


In [49]:
# Display TF-IDF values for the first training sentence

idx2feature = {i: feature for feature, i in vectorizer.vocabulary_.items()}

sample_index = 0
row = X_train[sample_index]

print("Sentence:")
print(train_df['text'].iloc[sample_index])

print("\nLanguage label:")
print(train_df['label'].iloc[sample_index])

print("\nNon-zero TF-IDF feature values (first 30):")
for col_idx, value in zip(row.indices[:30], row.data[:30]):
    print(f"{idx2feature[col_idx]}: {value:.4f}")

Sentence:
बताया कि जिन डीलरों द्वारा खाद्यान्न का उठाव कर लिया गया है ।

Language label:
hin_Deva

Non-zero TF-IDF feature values (first 30):
C_्न क: 0.1092
C_कर ल: 0.1092
C_र लि: 0.1083
C_यान्: 0.1083
C_ान्न: 0.1074
C_का उ: 0.1074
W_बताया▁कि: 0.1066
C_व कर: 0.1059
C_ों द: 0.1052
C_ जिन: 0.1045
C_ाद्य: 0.1038
C_ि जि: 0.1038
C_उठ: 0.1038
C_खाद: 0.1038
C_ लिय: 0.1038
C_ उठ: 0.1038
W_बताया: 0.1026
C_बताय: 0.1015
C_ाव क: 0.0999
W_द्वारा: 0.0985
C_जिन: 0.0981
C_लर: 0.0977
C_ताया: 0.0969
W_गया: 0.0965
C_गया : 0.0965
C_ाद्: 0.0958
C_ताय: 0.0950
C_ा गय: 0.0950
C_ा खा: 0.0947
C_न का: 0.0944


In [53]:
class CustomLogisticRegression:
    def __init__(self, n_features, n_classes, lr=0.5, reg_lambda=1e-4,
                 n_epochs=20, batch_size=256, seed=42):
        self.lr = lr
        self.reg_lambda = reg_lambda
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        rng = np.random.RandomState(seed)
        self.W = rng.normal(0, 0.01, size=(n_features, n_classes))
        self.b = np.zeros(n_classes)

    @staticmethod
    def _softmax(Z):
        Z = Z - Z.max(axis=1, keepdims=True)
        expZ = np.exp(Z)
        return expZ / expZ.sum(axis=1, keepdims=True)

    def fit(self, X, y, X_val=None, y_val=None, verbose=True):
        n_samples, n_classes = X.shape[0], self.W.shape[1]
        Y_onehot = np.zeros((n_samples, n_classes))
        Y_onehot[np.arange(n_samples), y] = 1

        for epoch in range(self.n_epochs):
            perm = np.random.permutation(n_samples)
            total_loss = 0.0
            for start in range(0, n_samples, self.batch_size):
                batch_idx = perm[start:start + self.batch_size]
                Xb = X[batch_idx]
                Yb = Y_onehot[batch_idx]

                Z = Xb.dot(self.W) + self.b
                P = self._softmax(Z)

                loss = -np.sum(Yb * np.log(P + 1e-12)) / Xb.shape[0]
                loss += 0.5 * self.reg_lambda * np.sum(self.W ** 2)
                total_loss += loss * Xb.shape[0]

                grad_Z = (P - Yb) / Xb.shape[0]
                grad_W = Xb.T.dot(grad_Z) + self.reg_lambda * self.W
                grad_b = grad_Z.sum(axis=0)

                self.W -= self.lr * grad_W
                self.b -= self.lr * grad_b

            if verbose:
                msg = f"Epoch {epoch+1}/{self.n_epochs} - loss: {total_loss/n_samples:.4f}"
                if X_val is not None:
                    val_acc = (self.predict(X_val) == y_val).mean()
                    msg += f" - val_acc: {val_acc:.4f}"
                print(msg)

    def predict_proba(self, X):
        Z = X.dot(self.W) + self.b
        return self._softmax(Z)

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

In [51]:
model = CustomLogisticRegression(
    n_features=X_train.shape[1],
    n_classes=N_CLASSES,
    lr=0.5,
    reg_lambda=1e-4,
    n_epochs=20,
    batch_size=256,
    seed=42
)

model.fit(X_train, y_train, X_val=X_val, y_val=y_val)

Epoch 1/20 - loss: 3.0714 - val_acc: 0.8919
Epoch 2/20 - loss: 2.9283 - val_acc: 0.9255
Epoch 3/20 - loss: 2.7898 - val_acc: 0.9118
Epoch 4/20 - loss: 2.6560 - val_acc: 0.9143
Epoch 5/20 - loss: 2.5277 - val_acc: 0.9273
Epoch 6/20 - loss: 2.4051 - val_acc: 0.9224
Epoch 7/20 - loss: 2.2891 - val_acc: 0.9304
Epoch 8/20 - loss: 2.1796 - val_acc: 0.9342
Epoch 9/20 - loss: 2.0769 - val_acc: 0.9304
Epoch 10/20 - loss: 1.9811 - val_acc: 0.9329
Epoch 11/20 - loss: 1.8921 - val_acc: 0.9348
Epoch 12/20 - loss: 1.8095 - val_acc: 0.9317
Epoch 13/20 - loss: 1.7332 - val_acc: 0.9354
Epoch 14/20 - loss: 1.6627 - val_acc: 0.9398
Epoch 15/20 - loss: 1.5977 - val_acc: 0.9410
Epoch 16/20 - loss: 1.5378 - val_acc: 0.9398
Epoch 17/20 - loss: 1.4826 - val_acc: 0.9398
Epoch 18/20 - loss: 1.4316 - val_acc: 0.9441
Epoch 19/20 - loss: 1.3846 - val_acc: 0.9447
Epoch 20/20 - loss: 1.3412 - val_acc: 0.9441


In [52]:
def confusion_matrix(y_true, y_pred, n_classes):
    cm = np.zeros((n_classes, n_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

def precision_recall_f1_per_class(cm):
    n_classes = cm.shape[0]
    precisions, recalls, f1s = [], [], []
    for c in range(n_classes):
        tp = cm[c, c]
        fp = cm[:, c].sum() - tp
        fn = cm[c, :].sum() - tp
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * precision * recall / (precision + recall)
              if (precision + recall) > 0 else 0.0)
        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)
    return precisions, recalls, f1s

def macro_f1_score(y_true, y_pred, n_classes):
    cm = confusion_matrix(y_true, y_pred, n_classes)
    precisions, recalls, f1s = precision_recall_f1_per_class(cm)
    return np.mean(f1s), precisions, recalls, f1s, cm

def accuracy_score(y_true, y_pred):
    return np.mean(np.array(y_true) == np.array(y_pred))

y_pred_test = model.predict(X_test)
macro_f1, precisions, recalls, f1s, cm = macro_f1_score(y_test, y_pred_test, N_CLASSES)

print(f"Test Accuracy: {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Test Macro-F1: {macro_f1:.4f}\n")

print(f"{'Language':<20}{'Precision':>10}{'Recall':>10}{'F1':>10}")
for i in range(N_CLASSES):
    print(f"{idx2label[i]:<20}{precisions[i]:>10.4f}{recalls[i]:>10.4f}{f1s[i]:>10.4f}")

Test Accuracy: 0.9366
Test Macro-F1: 0.9337

Language             Precision    Recall        F1
asm_Beng                0.9859    1.0000    0.9929
ben_Beng                0.9857    0.9857    0.9857
brx_Deva                0.9800    0.7000    0.8167
doi_Deva                0.9538    0.8857    0.9185
gom_Deva                0.7576    0.3571    0.4854
guj_Gujr                1.0000    1.0000    1.0000
hin_Deva                0.6939    0.9714    0.8095
kan_Knda                1.0000    1.0000    1.0000
kas_Arab                1.0000    0.9857    0.9928
khasi                   0.9589    1.0000    0.9790
mai_Deva                0.9242    0.8714    0.8971
mal_Mlym                1.0000    1.0000    1.0000
mar_Deva                0.6214    0.9143    0.7399
mni_Mtei                1.0000    0.9714    0.9855
npi_Deva                0.9565    0.9429    0.9496
ory_Orya                1.0000    1.0000    1.0000
pan_Guru                1.0000    1.0000    1.0000
san_Deva                0.9200    0.9